<a href="https://colab.research.google.com/github/DuhranDuhran/AI-tool-calling/blob/main/Multi_Tool_Intent_Router.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multi-Tool Intent Router

In [2]:
# Step 1: Install the official Google GenAI SDK
!pip install -q -U google-genai

import os
from google import genai
from google.genai import types
from google.colab import userdata

# Step 2: Initialize the Gemini Client
api_key = userdata.get("Gemini_API_Key1")
client = genai.Client(api_key=api_key)


# Step 3: Define multiple target Python functions (Tools)
def get_order_status(order_id: str) -> str:
    """Retrieves current tracking status for a given customer order ID.

    Args:
        order_id: The unique order identifier string (e.g., ORD-10294).
    """
    # Mock data lookup
    mock_database = {
        "ORD-10294": "In Transit - Expected Delivery Tomorrow by 5:00 PM",
        "ORD-88231": "Delivered yesterday at front porch",
    }
    return mock_database.get(order_id, f"Order {order_id} not found in database.")


def calculate_shipping_cost(weight_lbs: float, destination_zip: str) -> str:
    """Calculates shipping charges based on package weight and destination ZIP code.

    Args:
        weight_lbs: Weight of the package in pounds.
        destination_zip: 5-digit destination US ZIP code string.
    """
    base_rate = 5.00
    total_cost = base_rate + (weight_lbs * 1.50)
    return f"${total_cost:.2f} (Standard Ground to {destination_zip})"


def search_knowledge_base(query: str) -> str:
    """Searches the help desk knowledge base for general policies and FAQs.

    Args:
        query: Search query string regarding policies, returns, or company info.
    """
    return "Standard Return Policy: Items can be returned within 30 days of receipt in original packaging."


# Step 4: Test Intent Routing across different prompts

# Test Prompt 1: Should dynamically route to get_order_status
prompt_1 = "Can you tell me where package ORD-10294 is right now?"

response_1 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_1,
    config=types.GenerateContentConfig(
        tools=[get_order_status, calculate_shipping_cost, search_knowledge_base]
    )
)

print("--- TEST 1: Order Lookup Prompt ---")
print(f"User Query: {prompt_1}")
print(f"Gemini Routing Output:\n{response_1.text}\n")


# Test Prompt 2: Should dynamically route to calculate_shipping_cost
prompt_2 = "How much will it cost to send an 8.5 lb box to 90210?"

response_2 = client.models.generate_content(
    model="gemini-3.6-flash",
    contents=prompt_2,
    config=types.GenerateContentConfig(
        tools=[get_order_status, calculate_shipping_cost, search_knowledge_base]
    )
)

print("--- TEST 2: Shipping Cost Prompt ---")
print(f"User Query: {prompt_2}")
print(f"Gemini Routing Output:\n{response_2.text}\n")

--- TEST 1: Order Lookup Prompt ---
User Query: Can you tell me where package ORD-10294 is right now?
Gemini Routing Output:
Order ORD-10294 is currently **In Transit** and is expected to be delivered by **tomorrow at 5:00 PM**.

--- TEST 2: Shipping Cost Prompt ---
User Query: How much will it cost to send an 8.5 lb box to 90210?
Gemini Routing Output:
It will cost **$17.75** to send an 8.5 lb package to 90210 via Standard Ground shipping.

